Results: Galaxies and Fits
===========================

This tutorial inspects an inferred model using galaxies inferred by the non-linear search.
This allows us to visualize and interpret its results.

The galaxies and fit API is described fully in the guides:

 - `autolens_workspace/*/guides/tracer.ipynb`
 - `autolens_workspace/*/guides/fit.ipynb`
 - `autolens_workspace/*/guides/galaxies.ipynb`

This result example only explains specific functionality for using a `Result` object to inspect galaxies or a fit
and therefore you should read these guides in detail first.

__Contents__

**Units:** In this example, all quantities use the source code's internal unit coordinates, with spatial.
**Data Structures:** Arrays inspected in this example use bespoke data structures for storing arrays, grids, vectors and.
**Model Fit:** Perform the model-fit using the search and analysis.
**Max Likelihood Tracer:** As seen elsewhere in the workspace, the result contains a `max_log_likelihood_tracer` which we can.
**Refitting:** Using the API introduced in the first tutorial, we can also refit the data locally.
**Samples API:** In the first results tutorial, we used `Samples` objects to inspect the results of a model.
**Max Likelihood Fit:** As seen elsewhere in the workspace, the result contains a `max_log_likelihood_fit` which we can.
**Wrap Up:** Summary of the script and next steps.

__Units__

In this example, all quantities use the source code's internal unit coordinates, with spatial coordinates in
arc seconds, luminosities in electrons per second and mass quantities (e.g. convergence) are dimensionless.

The guide `guides/units_and_cosmology.ipynb` illustrates how to convert these quantities to physical units like
kiloparsecs, magnitudes and solar masses.

__Data Structures__

Arrays inspected in this example use bespoke data structures for storing arrays, grids,
vectors and other 1D and 2D quantities. These use the `slim` and `native` API to toggle between representing the
data in 1D numpy arrays or high dimension numpy arrays.

This tutorial will only use the `slim` properties which show results in 1D numpy arrays of
shape [total_unmasked_pixels]. This is a slimmed-down representation of the data in 1D that contains only the
unmasked data points

These are documented fully in the `autolens_workspace/*/guides/data_structures.ipynb` guide.

__Start Here Notebook__

If any code in this script is unclear, refer to the `results/start_here.ipynb` notebook.

In [ ]:

from autoconf import jax_wrapper  # Sets JAX environment before other imports

# %matplotlib inline
# from pyprojroot import here
# workspace_path = str(here())
# %cd $workspace_path
# print(f"Working Directory has been set to `{workspace_path}`")

import numpy as np
from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Model Fit__

To illustrate results, we need to perform a model-fit in order to create a `Result` object.

The code below performs a model-fit using nautilus. 

You should be familiar with modeling already, if not read the `modeling/start_here.py` script before reading this one.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if not dataset_path.exists():
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/features/no_lens_light/simulator.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)


model = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(al.Galaxy, redshift=0.5, mass=al.mp.Isothermal),
        source=af.Model(al.Galaxy, redshift=1.0, bulge=bulge, disk=None),
    ),
)

search = af.Nautilus(
    path_prefix=Path("results_folder"),
    name="results",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
)

analysis = al.AnalysisImaging(dataset=dataset, use_jax=True)

result = search.fit(model=model, analysis=analysis)


__Max Likelihood Tracer__

As seen elsewhere in the workspace, the result contains a `max_log_likelihood_tracer` which we can visualize.

In [ ]:
tracer = result.max_log_likelihood_tracer

aplt.subplot_tracer(tracer=tracer, grid=mask.derive_grid.all_false)

__Refitting__

Using the API introduced in the first tutorial, we can also refit the data locally. 

This allows us to inspect how the tracer changes for models with similar log likelihoods. We create and plot
the tracer of the 100th last accepted model by Nautilus.

In [ ]:
samples = result.samples

instance = samples.from_sample_index(sample_index=-10)

# Input to FitImaging to solve for linear light profile intensities, see `start_here.py` for details.
tracer = al.Tracer(galaxies=instance.galaxies)
fit = al.FitImaging(dataset=dataset, tracer=tracer)
tracer = fit.tracer_linear_light_profiles_to_light_profiles

aplt.subplot_tracer(tracer=tracer, grid=mask.derive_grid.all_false)

__Samples API__

In the first results tutorial, we used `Samples` objects to inspect the results of a model.

We saw how these samples created instances, which include a `galaxies` property that mains the API of the `Model`
creates above (e.g. `galaxies.source.bulge`). 

We can also use this instance to extract individual components of the model.

In [ ]:
samples = result.samples

ml_instance = samples.max_log_likelihood()

# Input to FitImaging to solve for linear light profile intensities, see `start_here.py` for details.
tracer = al.Tracer(galaxies=instance.galaxies)
fit = al.FitImaging(dataset=dataset, tracer=tracer)
tracer = fit.tracer_linear_light_profiles_to_light_profiles

bulge = tracer.galaxies[-1].bulge

bulge_image_2d = bulge.image_2d_from(grid=dataset.grid)
print(bulge_image_2d.slim[0])

aplt.plot_array(array=bulge.image_2d_from(grid=dataset.grid), title="Image")

In fact, if we create a `Tracer` from an instance (which is how `result.max_log_likelihood_tracer` is created) we
can choose whether to access its attributes using each API: 

In [ ]:
tracer = result.max_log_likelihood_tracer
print(tracer.galaxies[-1].bulge)

__Max Likelihood Fit__

As seen elsewhere in the workspace, the result contains a `max_log_likelihood_fit` which we can visualize.

In [ ]:
fit = result.max_log_likelihood_fit

aplt.subplot_fit_imaging(fit=fit)


__Refitting__

Using the API introduced in the first tutorial, we can also refit the data locally. 

This allows us to inspect how the fit changes for models with similar log likelihoods. Below, we refit and plot
the fit of the 100th last accepted model by Nautilus.

In [ ]:
samples = result.samples

instance = samples.from_sample_index(sample_index=-10)

tracer = al.Tracer(galaxies=instance.galaxies)

fit = al.FitImaging(dataset=dataset, tracer=tracer)

aplt.subplot_fit_imaging(fit=fit)

__Wrap Up__

We have learnt how to extract individual planes, galaxies, light and mass profiles from the tracer that results from
a model-fit and use these objects to compute specific quantities of each component.